In [ ]:
pip install transformers timm torchvision tqdm pandas

In [ ]:
import os
import torch
import numpy as np
import cv2
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

model.eval()

print("Model loaded on:", device)

In [ ]:
def js_divergence(p, q):
    eps = 1e-8
    p = p.flatten() + eps
    q = q.flatten() + eps
    
    p = p / p.sum()
    q = q / q.sum()
    
    m = 0.5 * (p + q)
    
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    
    return 0.5 * (kl_pm + kl_qm)


def kl_divergence(p, q):
    eps = 1e-8
    p = p.flatten() + eps
    q = q.flatten() + eps
    
    p = p / p.sum()
    q = q / q.sum()
    
    return np.sum(p * np.log(p / q))

In [ ]:
def process_image(image_path, saliency_path):

    # ---- Load image ----
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    # ---- Generate caption ----
    with torch.no_grad():
        generated_ids = model.generate(**inputs)

    caption = processor.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    # ---- Vision encoder ----
    with torch.no_grad():
        encoder_outputs = model.vision_model(
            pixel_values=inputs["pixel_values"],
            return_dict=True
        )

    image_embeds = encoder_outputs.last_hidden_state

    # ---- Text decoder ----
    with torch.no_grad():
        decoder_outputs = model.text_decoder(
            input_ids=generated_ids,
            encoder_hidden_states=image_embeds,
            output_attentions=True,
            return_dict=True
        )

    cross_attentions = decoder_outputs.cross_attentions

    # ---- Extract spatial attention (24x24) ----
    last_layer = cross_attentions[-1]
    last_layer = last_layer.mean(dim=1).squeeze(0)
    last_layer = last_layer[:, 1:]  # remove CLS

    token_attention_maps = []

    for t in range(last_layer.shape[0]):
        token_attn = last_layer[t]
        token_attn = token_attn.reshape(24, 24)

        token_attn = token_attn - token_attn.min()
        token_attn = token_attn / (token_attn.sum() + 1e-8)

        token_attention_maps.append(token_attn.cpu().numpy())

    token_attention_maps = np.stack(token_attention_maps)

    # ---- Load saliency map ----
    saliency = cv2.imread(saliency_path, cv2.IMREAD_GRAYSCALE)
    saliency = cv2.resize(saliency, (24, 24)).astype(np.float32)

    saliency = saliency - saliency.min()
    saliency = saliency / (saliency.sum() + 1e-8)

    # ---- Compute divergence ----
    js_values = []
    kl_values = []

    for t in range(token_attention_maps.shape[0]):
        model_attn = token_attention_maps[t]

        js = js_divergence(model_attn, saliency)
        kl = kl_divergence(model_attn, saliency)

        js_values.append(js)
        kl_values.append(kl)

    return {
    "caption": caption,
    "num_tokens": token_attention_maps.shape[0],
    "avg_JS": float(np.mean(js_values)),
    "avg_KL": float(np.mean(kl_values)),
    "token_JS_list": js_values,
    "token_KL_list": kl_values
}

In [ ]:
image_dir = "Enter the path"
saliency_dir = "Enter the path"

In [ ]:
results = []

image_files = sorted(os.listdir(image_dir))[:50]  # start with 50

for file in tqdm(image_files):

    image_path = os.path.join(image_dir, file)
    saliency_path = os.path.join(
        saliency_dir,
        file.replace(".jpg", ".png")
    )

    if not os.path.exists(saliency_path):
        continue

    output = process_image(image_path, saliency_path)
    output["image_id"] = file

    results.append(output)

print("Processed:", len(results))

In [ ]:
df = pd.DataFrame(results)
df.head()

In [ ]:
df.to_csv("blip_attention_divergence_results.csv", index=False)

In [ ]:
import json

annotation_file = "Enter the Path"

with open(annotation_file, 'r') as f:
    coco_data = json.load(f)

# Build image_id → reference captions dictionary
gt_dict = {}

for ann in coco_data["annotations"]:
    image_id = ann["image_id"]
    caption = ann["caption"]
    
    if image_id not in gt_dict:
        gt_dict[image_id] = []
    
    gt_dict[image_id].append(caption)

print("Loaded ground truth captions.")
print("Total images with captions:", len(gt_dict))

In [ ]:
pip install nltk

In [ ]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.translate.bleu_score import sentence_bleu
from nltk.tokenize import word_tokenize

In [ ]:
def compute_bleu(generated_caption, references):
    gen_tokens = word_tokenize(generated_caption.lower())
    ref_tokens = [word_tokenize(ref.lower()) for ref in references]
    
    return sentence_bleu(ref_tokens, gen_tokens)

In [ ]:
bleu_scores = []

for idx, row in df.iterrows():
    
    # Extract numeric COCO ID
    image_id = int(row["image_id"].split("_")[-1].split(".")[0])
    
    if image_id in gt_dict:
        references = gt_dict[image_id]
        bleu = compute_bleu(row["caption"], references)
    else:
        bleu = 0
    
    bleu_scores.append(bleu)

df["BLEU"] = bleu_scores

df.head()

In [ ]:
df["correct"] = df["BLEU"] > 0.3

df["correct"].value_counts()

In [ ]:
correct_js = df[df["correct"]]["avg_JS"]
incorrect_js = df[~df["correct"]]["avg_JS"]

print("Correct JS mean:", correct_js.mean())
print("Incorrect JS mean:", incorrect_js.mean())

In [ ]:
correct_kl = df[df["correct"]]["avg_KL"]
incorrect_kl = df[~df["correct"]]["avg_KL"]

print("Correct KL mean:", correct_kl.mean())
print("Incorrect KL mean:", incorrect_kl.mean())

In [ ]:
from scipy.stats import ttest_ind

t_js = ttest_ind(correct_js, incorrect_js, equal_var=False)
t_kl = ttest_ind(correct_kl, incorrect_kl, equal_var=False)

print("JS t-test:", t_js)
print("KL t-test:", t_kl)

In [ ]:
df = pd.DataFrame(results)
df.columns

In [ ]:
df["max_JS"] = df["token_JS_list"].apply(max)
df["max_KL"] = df["token_KL_list"].apply(max)

In [ ]:
correct_max_js = df[df["correct"]]["max_JS"]
incorrect_max_js = df[~df["correct"]]["max_JS"]

print("Correct max_JS mean:", correct_max_js.mean())
print("Incorrect max_JS mean:", incorrect_max_js.mean())

In [ ]:
labels = pd.read_csv("manual_labels.csv")

df["image_id"] = df["image_id"].str.replace(".jpg", "", regex=False)
df = df.merge(labels[["image_id", "error_type"]], on="image_id")

In [ ]:
print("DF image_id sample:")
print(df["image_id"].iloc[0], repr(df["image_id"].iloc[0]))

print("\nLabels image_id sample:")
print(labels["image_id"].iloc[0], repr(labels["image_id"].iloc[0]))

In [ ]:
print(df["image_id"].head())
print(labels["image_id"].head())

In [ ]:
print(df)

In [ ]:
df["has_error"] = df["error_type"] != 0

In [ ]:
correct_js = df[df["has_error"] == False]["avg_JS"]
error_js = df[df["has_error"] == True]["avg_JS"]

print("Correct JS mean:", correct_js.mean())
print("Error JS mean:", error_js.mean())

In [ ]:
from scipy.stats import ttest_ind

t = ttest_ind(correct_js, error_js, equal_var=False)
print(t)

In [ ]:
correct_max = df[df["has_error"] == False]["max_JS"]
error_max = df[df["has_error"] == True]["max_JS"]

print("Correct max_JS:", correct_max.mean())
print("Error max_JS:", error_max.mean())

In [ ]:
# Reload labels fresh
labels = pd.read_csv("manual_labels.csv")

# Normalize IDs
df["image_id"] = df["image_id"].str.replace(".jpg", "", regex=False).str.strip()
labels["image_id"] = labels["image_id"].str.strip()

# Merge full label info
df = df.merge(
    labels[["image_id", "error_type", "primary_error_token"]],
    on="image_id",
    how="inner"
)

print(len(df))
print(df.columns)

In [ ]:
# Keep only one error_type column
df["error_type"] = df["error_type_y"]

# Drop the duplicates
df = df.drop(columns=["error_type_x", "error_type_y"])

print(df.columns)

In [ ]:
import re
import numpy as np

error_token_js = []
non_error_mean_js = []

for idx, row in df.iterrows():
    
    if row["error_type"] == 0:
        error_token_js.append(np.nan)
        non_error_mean_js.append(np.nan)
        continue
    
    caption = row["caption"].lower()
    js_list = row["token_JS_list"]
    
    # Clean caption (remove punctuation)
    clean_caption = re.sub(r'[^\w\s]', '', caption)
    tokens = clean_caption.split()
    
    target = str(row["primary_error_token"]).lower()
    target = re.sub(r'[^\w\s]', '', target)
    
    # For multi-word error tokens, match first word
    target_words = target.split()
    
    token_index = None
    
    for i, tok in enumerate(tokens):
        if tok in target_words:
            token_index = i
            break
    
    if token_index is not None and token_index < len(js_list):
        error_js = js_list[token_index]
        other_js = [js_list[i] for i in range(len(js_list)) if i != token_index]
        
        error_token_js.append(error_js)
        non_error_mean_js.append(np.mean(other_js))
    else:
        error_token_js.append(np.nan)
        non_error_mean_js.append(np.nan)

df["error_token_JS"] = error_token_js
df["non_error_mean_JS"] = non_error_mean_js

In [ ]:
print(df[["caption", "primary_error_token", "error_token_JS"]])

print("\nNaN counts:")
print(df[["caption", "primary_error_token", "error_token_JS"]].isna().sum())

In [ ]:
error_rows = df[df["error_type"] != 0]

print("Number of error cases:", len(error_rows))

print("Mean JS at error token:",
      error_rows["error_token_JS"].mean())

print("Mean JS at other tokens:",
      error_rows["non_error_mean_JS"].mean())

In [ ]:
from scipy.stats import ttest_rel

valid = error_rows.dropna(subset=["error_token_JS"])

t = ttest_rel(valid["error_token_JS"],
              valid["non_error_mean_JS"])

print(t)

In [ ]:
import numpy as np

diff = valid["error_token_JS"] - valid["non_error_mean_JS"]
cohens_d = diff.mean() / diff.std()
print("Cohen's d:", cohens_d)